# In-Depth Data Science Evaluation

This notebook provides a deep analytical dive into the actual model metrics logged during training. 

> **Methodology Note**: As the raw gigabyte image dataset and softmax probability arrays are purposely omitted from this repository due to PHI protection and GitHub size constraints, this analysis derives strictly from the preserved classification report JSONs and confusion matrices representing the verified validation subset.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Class-Wise Performance Analysis
We evaluate how the model behaves across individual classes using the InceptionV3 verified logs.

In [ ]:
cr_path = 'inception_79%/classification_report.json'
if os.path.exists(cr_path):
    with open(cr_path, 'r') as f:
        report = json.load(f)
    
    # Extract Real Data
    metrics = {
        'Class': ['Abnormal', 'Normal'],
        'Precision': [report['Abnormal']['precision'], report['Normal']['precision']],
        'Recall': [report['Abnormal']['recall'], report['Normal']['recall']],
        'F1-Score': [report['Abnormal']['f1-score'], report['Normal']['f1-score']],
        'Support': [report['Abnormal']['support'], report['Normal']['support']]
    }
    df_metrics = pd.DataFrame(metrics)
    display(df_metrics)
    
    # Plotting
    df_metrics.set_index('Class')[['Precision', 'Recall', 'F1-Score']].plot(kind='bar', figsize=(8, 5), colormap='Set2')
    plt.title('Verified Class-Wise Metrics (InceptionV3)')
    plt.ylim(0.6, 1.0)
    plt.xticks(rotation=0)
    plt.show()
else:
    print("Metrics JSON not found locally.")

### Conclusion on Class Bias:
The model exhibits asymmetric behavior:
- **Abnormal Recall is High (0.816)**: The model is aggressive in catching disease.
- **Normal Precision is High (0.858)**: When the model says 'Normal', it is usually correct.
 
This asymmetry is ideal for a screening triage tool where False Negatives are the primary danger.

## 2. Threshold Sensitivity & Confidence Analysis

In a live deployment, models do not output binary classes; they output a sigmoid probability $P(Y=1|X) \in [0, 1]$. 

> **Limitation Documentation**: Because the dense probability tensor (`val_preds`) was not logged permanently to disk (only the argmaxed `confusion_matrix.json` was kept), we cannot dynamically replot a Precision-Recall Curve based on threshold shifts.

### Theoretical Optimization Framework
If the probabilities were preserved, we would run a threshold sensitivity scan:
1. **Default Threshold (0.50)**: Yields the current 64 False Negatives.
2. **Conservative Threshold (0.35)**: Classifying anything with >35% pathology risk as Abnormal. This would increase False Positives (over 113) but severely drop False Negatives (below 64), which is fundamentally required for Phase 1 clinical screening protocols.

## 3. Deployment Suitability Breakdown (InceptionV3 vs ResNet152)

**InceptionV3**:
- Highly modular blocks allow strong spatial hierarchy learning (identifying scattered microaneurysms).
- Empirically stable logs. 
- Clear deployment pathway on simple GPUs.

**ResNet152 + Quantum Analysis Layer**:
- Massive parameter size makes it theoretically stronger for deep hierarchical features.
- Quantum gradient simulations (PennyLane) introduce extreme computational overhead, making edge-device processing unfeasible. It serves purely as an experimental academic anchor rather than a clinical tool.